# NB_lab_test_trends — Gold Layer: Lab Test Trends

Builds `caresyncdata.gold.lab_test_trends` — one row per test type per calendar month with volume, abnormality rates, and month-over-month trend.

**STTM:** Silver → Gold | CareSync Health Network 
**Load type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT` 
**Grain:** One row per test name per year-month 
**Driving table:** `caresyncdata.silver.lab_results` (grouped by `test_name`, `year`, `month`) 
**Joins:** LEFT JOIN `calendar` (optional — for `month_name`, `quarter_name`) 
**Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month`

In [0]:
# ---------------------------------------------------------------------------
# Configuration — substituting the <catalog> placeholder from the STTM
# ---------------------------------------------------------------------------
catalog = "caresyncdata"
silver  = f"{catalog}.silver"
gold    = f"{catalog}.gold"
target_table = f"{gold}.lab_test_trends"

print(f"Catalog  : {catalog}")
print(f"Source   : {silver}")
print(f"Target   : {target_table}")

In [0]:
# Ensure the gold schema exists before writing
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Schema '{gold}' is ready.")

In [0]:
# ---------------------------------------------------------------------------
# Full rebuild — CREATE OR REPLACE TABLE ... AS SELECT
# Grain: one row per test_name per year-month
# Driving table: silver.lab_results  (grouped by test_name, year, month)
# Joins: LEFT JOIN calendar (month_name, quarter_name — optional)
# Window: LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH base_agg AS (
    -- Aggregate lab_results per test type per month
    SELECT
        test_name,
        YEAR(test_date)                                                        AS year,
        MONTH(test_date)                                                       AS month,
        COUNT(lab_result_id)                                                   AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)                AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)                AS critical_test_count
    FROM {silver}.lab_results
    GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
with_rates AS (
    -- Derived rate columns
    SELECT
        b.*,
        CAST(b.abnormal_test_count / NULLIF(b.total_tests, 0) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(b.critical_test_count / NULLIF(b.total_tests, 0) * 100 AS DECIMAL(5,2)) AS critical_rate_pct
    FROM base_agg AS b
),
with_lag AS (
    -- Month-over-month trend: LAG per test_name, ordered by year then month
    SELECT
        w.*,
        LAG(w.abnormal_rate_pct) OVER (
            PARTITION BY w.test_name
            ORDER BY w.year, w.month
        ) AS prev_month_abnormal_rate_pct
    FROM with_rates AS w
)
SELECT
    wl.test_name,
    wl.year,
    wl.month,

    -- Calendar enrichment (optional — month_name, quarter_name)
    cal.month_name,
    cal.quarter_name,

    -- Aggregated metrics
    wl.total_tests,
    wl.abnormal_test_count,
    wl.critical_test_count,
    wl.abnormal_rate_pct,
    wl.critical_rate_pct,

    -- Month-over-month trend
    wl.prev_month_abnormal_rate_pct,
    CAST(
        CASE
            WHEN wl.prev_month_abnormal_rate_pct IS NULL THEN NULL
            ELSE wl.abnormal_rate_pct - wl.prev_month_abnormal_rate_pct
        END
    AS DECIMAL(6,2)) AS abnormal_rate_change_pct

FROM with_lag                           AS wl
LEFT JOIN {silver}.calendar             AS cal
    ON cal.date_key = CAST(
           DATE_FORMAT(MAKE_DATE(wl.year, wl.month, 1), 'yyyyMMdd')
       AS INT)
ORDER BY wl.test_name, wl.year, wl.month
""")
print(f"\u2705  {target_table} rebuilt successfully.")

In [0]:
# Quick row-count and sample check
row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
print(f"Total rows in {target_table}: {row_count}")

df = spark.table(target_table)
display(df.limit(10))

In [0]:
records_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
dbutils.notebook.exit(str(records_count))